# Script Cue Building with SQLite Database Integration (SQLModel)

This notebook parses Fountain screenplay format and generates character mute/unmute cues, storing them in a SQLite database using SQLModel ORM.

In [ ]:
from fountain import fountain
from qlab.database_sqlmodel import CueDatabase
from pathlib import Path

## Load and Parse Script

In [2]:
with open('../seussical/scripts/seussical.fountain', 'r') as file:
    f = fountain.Fountain(file.read())
    f.parse()

print(f"Loaded {len(f.elements)} script elements")

Loaded 5718 script elements


## Helper Function: Predict Future Speaking

In [3]:
def speaks_within(book, character, n: int = 7):
    """Check if character speaks within next n dialogue blocks or before scene change.
    
    Args:
        book: List of script elements to search
        character: Character name to look for
        n: Number of dialogue blocks to look ahead
    
    Returns:
        True if character speaks within window, False otherwise
    """
    dialogues = 0
    for i, element in enumerate(book):
        if dialogues >= n:
            return False
        if element.element_type == 'Scene Heading':
            return False
        if element.element_type == 'Character':
            if element.element_text == character:
                return True
            dialogues += 1
    return False

## Generate Cues from Script

In [4]:
script = f.elements

cues = []
active = set()

for i, element in enumerate(script):
    # For every Character element
    if element.element_type == 'Character':
        character_name = element.element_text
        
        # Get preview of dialogue line
        line_preview = script[i+1].element_text[:30] + '...' if i+1 < len(script) else ''
        
        # If they aren't active, unmute
        if character_name not in active:
            cue_type = 'unmute'
            cue_name = f'unmute {character_name.title()}'
            print(f"{cue_name}: {line_preview}")
            active.add(character_name)
            cues.append((cue_type, character_name, line_preview))
        
        # If they won't speak again soon, mute them
        if not speaks_within(script[i+1:], character_name):
            cue_type = 'mute'
            cue_name = f'mute {character_name.title()}'
            line_end = '...' + script[i+1].element_text[-30:] if i+1 < len(script) else ''
            print(f"{cue_name}: {line_end}")
            active.remove(character_name)
            cues.append((cue_type, character_name, line_end))

print(f"\nGenerated {len(cues)} total cues")

unmute Boy: Now that is a very unusual hat...
unmute Cat In The Hat: I can see that you've got quit...
mute Cat In The Hat: ...AN THINK
WHEN YOU THINK ABOUT…
unmute Cat, All (Except Boy): SEUSS!
SEUSS!
SEUSS!
SEUSS!
SE...
mute Cat, All (Except Boy): ...! SEUSS! SEUSS! SEUSS!
SEUSS!!
unmute Cat & All (Except Boy): OH, THE THINKS YOU CAN THINK!
...
mute Cat & All (Except Boy): ...THINKS
CAN COME UP WITH A FEW!
mute Boy: ...OH, THE THINKS YOU CAN THINK!
unmute Cat & All: THINK A TRIP ON A SHIP
TO THE ...
mute Cat & All: ...N A SHIP
TO THE VIPPER OF VIPP
unmute Women: OR TO SOLLA SOLLEW…...
mute Women: ...OR TO SOLLA SOLLEW…
unmute Cat (Spoken): Think of beautiful Schlopp…...
mute Cat (Spoken): ...Think of beautiful Schlopp…
unmute Boy (Spoken): With a cherry on top!...
mute Boy (Spoken): ...With a cherry on top!
unmute Cat, Boy, All: YOU DON'T NEED AN EXCUSE!...
unmute Cat, Boy (Spoken): Oh, the thinks you can think...
mute Cat, Boy (Spoken): ...Oh, the thinks you can think
mute Cat, Boy,

## Save to CSV (Legacy Export)

In [6]:
from csv import writer

with open('cues.csv', 'w') as file:
    csv = writer(file)
    csv.writerow(['cue_type', 'character', 'line'])
    for cue in cues:
        csv.writerow(cue)

print(f"Saved {len(cues)} cues to cues.csv")

Saved 1446 cues to cues.csv


## Save to SQLite Database

Create or use an existing database to store the cues.

In [5]:
# Choose database - either create new or use existing
db_path = 'mix/seuss.sqlite'  # Change to use existing database

# You can also use the example database:
# db_path = 'mix/SheKillsMonsters.sqlite'

print(f"Using database: {db_path}")

Using database: mix/seuss.sqlite


In [ ]:
with CueDatabase(db_path, create_schema=True, init_config=True) as db:
    # Get starting point
    start_num, start_point = db.get_next_cue_number()
    print(f"Starting from cue {start_num}.{start_point}")
    
    added_cues = []
    
    for cue_type, character, line_preview in cues:
        # Try to get channel number from profiles
        print(f"Adding {cue_type} cue for {character.title()}")
        channel = db.get_channel_for_character(character.title())
        channels_str = str(channel) if channel else ''
        
        # Add the cue
        if cue_type == 'mute':
            point = db.add_mute_cue(
                character=character.title(),
                channels=channels_str,
                line_preview=line_preview,
                dca=1,  # You can customize which DCA to use
            )
        else:  # unmute
            point = db.add_unmute_cue(
                character=character.title(),
                channels=channels_str,
                line_preview=line_preview,
                dca=1,
            )
        
        added_cues.append(point)
    
    print(f"\nAdded {len(added_cues)} cues to database")
    print(f"Cue points: {added_cues[0]} to {added_cues[-1]}")

Starting from cue 2.0
Adding unmute cue for Boy
Adding unmute cue for Cat In The Hat
Adding mute cue for Cat In The Hat
Adding unmute cue for Cat, All (Except Boy)
Adding mute cue for Cat, All (Except Boy)
Adding unmute cue for Cat & All (Except Boy)
Adding mute cue for Cat & All (Except Boy)
Adding mute cue for Boy
Adding unmute cue for Cat & All
Adding mute cue for Cat & All
Adding unmute cue for Women
Adding mute cue for Women
Adding unmute cue for Cat (Spoken)
Adding mute cue for Cat (Spoken)
Adding unmute cue for Boy (Spoken)
Adding mute cue for Boy (Spoken)
Adding unmute cue for Cat, Boy, All
Adding unmute cue for Cat, Boy (Spoken)
Adding mute cue for Cat, Boy (Spoken)
Adding mute cue for Cat, Boy, All
Adding unmute cue for Horton
Adding mute cue for Horton
Adding unmute cue for Mr. And Mrs. Mayor
Adding mute cue for Mr. And Mrs. Mayor
Adding unmute cue for Gertrude
Adding mute cue for Gertrude
Adding unmute cue for All
Adding mute cue for All
Adding unmute cue for Mayzie
Adding 

## Verify Database Contents

In [ ]:
with CueDatabase(db_path, create_schema=False, init_config=False) as db:
    all_cues = db.get_all_cues()
    print(f"Total cues in database: {len(all_cues)}")
    
    # Show last 5 cues
    print("\nLast 5 cues:")
    for cue in all_cues[-5:]:
        print(f"  {cue.number}.{cue.point}: {cue.name}")

Total cues in database: 1447

Last 5 cues:
  1443.0: unmute Group 1 Group 2 Group 3 Group 4
  1444.0: mute Group 1 Group 2 Group 3 Group 4
  1445.0: unmute Boy
  1446.0: mute Boy
  1447.0: mute All


## Advanced: Add Cues with Custom Parameters

In [ ]:
# Example: Add a custom cue with specific DCA assignments
with CueDatabase(db_path, create_schema=False, init_config=False) as db:
    point = db.add_cue(
        name="Custom Scene Transition",
        dca_channels={1: "1,2,3", 8: "8"},  # DCA 1 has channels 1,2,3; DCA 8 has channel 8
        dca_labels={1: "Leads"},
        qlab_cue="s100",  # Reference to QLab sound cue
        colour=5,  # Custom color code
    )
    print(f"Added custom cue at point {point}")